# Eau et agriculture : qualité des eaux et usage des pesticides

**Problématique :** _à formuler_

Auteurs : _à compléter_

## 0. Imports et configuration

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.eau.recuperer_tous_indices import recuperer_tous_indices
from src.eau.nettoyer_indices import nettoyer_indices
from src.eau.rattacher_stations_zones_hydro import rattacher_stations_zones_hydro
from src.eau.agreger_eau_zones_hydro import agreger_eau_zones_hydro
from src.geo.constantes_geo import DEPARTEMENT_VERS_REGION
from src.geo.recuperer_departements import recuperer_departements
from src.geo.recuperer_zones_hydro import recuperer_zones_hydro
from src.agriculture.constantes_rpg import CATEGORIES_CULTURE, FILIERES_RPG
from src.zones_hydrographiques.associer_parcelles_zones_hydro import associer_parcelles_zones_hydro
from src.zones_hydrographiques.constantes_cultures_molecules import MOLECULES_PAR_GROUPE
from src.zones_hydrographiques.calculer_doses_pesticides_groupes import calculer_doses_pesticides_groupes
from src.pesticides.constantes_pesticides import LISTE_COLONNES_MOLECULES, MOLECULES
from src.zones_hydrographiques.agreger_champs_zones_hydro import agreger_champs_zones_hydro
from src.zones_hydrographiques.agreger_pesticides_zones_hydro import agreger_pesticides_zones_hydro
from src.zones_hydrographiques.fusionner_eau_cultures_pesticides import fusionner_eau_cultures_pesticides
from src.visualisation.calculer_gini import calculer_gini
from src.pesticides.preparer_pesticides import preparer_pesticides
from src.recuperer_release import recuperer_release
from src.visualisation.tracer_evolution_indices import tracer_evolution_indices
from src.visualisation.tracer_carte import tracer_carte
from src.visualisation.tracer_filieres_rpg import tracer_filieres_rpg
from src.visualisation.tracer_regions_rpg import tracer_regions_rpg
from src.visualisation.tracer_zoom_departements import tracer_zoom_departements
from src.visualisation.tracer_parcelles_coupees import tracer_parcelles_coupees

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e5e5e5",
})

## 1. Construction des trois bases

Chaque partie produit sa base, au format Parquet :

| Base | Maille | Source |
|---|---|---|
| `eau_zones_hydro.parquet` | zone hydrographique × année | Hub'Eau Hydrobiologie |
| `pesticides.parquet` | département × année × molécule | BNV-D (data.gouv.fr) |
| `parcelles_rpg_2023.parquet` | parcelle | RPG 2023 (IGN) |

Le territoire couvre les 94 départements de métropole continentale, sur 2013-2023. La Corse est exclue de tout le projet : le RPG y est trop incomplet.

Tout est recalculé à partir des sources, sauf les deux tables qui prennent plus de 5 minutes à calculer, parce qu'elles sont tirées de l'archive du RPG 2023 de l'IGN (2,75 Go) : `parcelles_rpg_2023.parquet` (partie 1.4) et `parcelles_zones_hydro.parquet` (partie 3.1). Elles sont lues dans la release [`donnees-v1`](https://github.com/raphaelcoic/Projet_python_data_science/releases/tag/donnees-v1) avec `recuperer_release`. Le code qui les produit reste dans `src/`.

La base eau est directement agrégée à la zone hydrographique : il faut donc d'abord les contours des zones (partie 1.1).

### 1.1 Fonds de carte et zones hydrographiques

- Les contours simplifiés des départements ([france-geojson](https://github.com/gregoiredavid/france-geojson)), projetés en Lambert 93 (EPSG:2154), la projection officielle de la métropole, servent aux cartes.
- Les 6 190 zones hydrographiques de la [BD TOPAGE](https://www.sandre.eaufrance.fr/atlas/srv/fre/catalog.search#/home) découpent la France en petits bassins versants. On retire les 80 zones de Corse. Elles servent à construire la base eau (partie 1.2) et au croisement (partie 3).

In [ ]:
departements = recuperer_departements().to_crs("EPSG:2154")

In [ ]:
zones_hydro = recuperer_zones_hydro()
zones_hydro = zones_hydro[~zones_hydro["code_zone"].str.startswith("12")]  # zones de Corse
zones_hydro.shape

### 1.2 Eau : `eau_zones_hydro.parquet`

La qualité d'une rivière se mesure aussi par les êtres vivants qu'on y trouve. Trois indices, calculés sur 2013-2023 :

| Indice | Ce qu'il observe | Échelle | Sensible à |
|---|---|---|---|
| **I2M2** | invertébrés (larves, vers, mollusques) | 0 à 1, **haut = bonne qualité** | pollutions multiples, dont les pesticides |
| **IBD** | diatomées (micro-algues) | 0 à 20, **haut = bonne qualité** | nutriments (engrais) et matière organique |
| **IPR** | poissons | 0 et plus, **bas = bonne qualité** | état physique de la rivière, pollutions |

**Récupération.** Les indices sur 2013-2023 sont demandés à l'API Hub'Eau, département par département, 8 départements à la fois (`recuperer_tous_indices`, environ 1 min 30). Chaque ligne donne aussi les coordonnées de sa station : la table des stations en est tirée.

In [ ]:
indices_bruts = recuperer_tous_indices()
stations = indices_bruts.drop_duplicates("code_station_hydrobio").rename(columns={"code_station_hydrobio": "code_station"})

print("indices bruts :", len(indices_bruts))
print("stations :", len(stations))

**Nettoyage** (`nettoyer_indices`). On retire les valeurs que le producteur ne qualifie pas de « correctes » (environ 9 % des lignes), les valeurs manquantes et les valeurs codées 999 de l'IPR (indice non calculable), et les doublons exacts. Quand une station a plusieurs valeurs d'un même indice le même jour (environ 4 400 cas), on garde leur moyenne : il reste une valeur par station, date et indice.

Après ce nettoyage, il ne reste aucune valeur manquante et toutes les valeurs sont dans l'échelle de leur indice (0 à 1 pour l'I2M2, 0 à 20 pour l'IBD). L'IPR n'a pas de borne haute : ses valeurs élevées (jusqu'à 165) signalent des rivières très dégradées, on les garde.

In [ ]:
indices = nettoyer_indices(indices_bruts)

print("indices après nettoyage :", len(indices))
indices.head()

**Rattachement aux zones** (`rattacher_stations_zones_hydro`). Chaque station ayant un indice va dans la zone hydrographique qui la contient, ou dans la plus proche si elle est juste hors limites. La zone est plus fine que le département et suit l'écoulement de l'eau, mais elle ne couvre que le bassin local : pour une station sur un grand cours d'eau, une partie de l'amont reste en dehors.

In [ ]:
stations_suivies = stations[stations["code_station"].isin(indices["code_station"])]
stations_zones_hydro = rattacher_stations_zones_hydro(stations_suivies, zones_hydro)

print("stations suivies :", len(stations_zones_hydro), "dans", stations_zones_hydro["code_zone"].nunique(), "zones")

**Agrégation** (`agreger_eau_zones_hydro`). Notons $x_{s,k,j}$ les $n_{s,k,t}$ mesures de l'indice $k$ à la station $s$ pendant l'année $t$. On fait la moyenne de chaque station sur l'année, puis la médiane des stations de la zone $z$ :

$$\bar{x}_{s,k,t} = \frac{1}{n_{s,k,t}} \sum_{j=1}^{n_{s,k,t}} x_{s,k,j} \qquad\qquad I_{z,k,t} = \underset{s \in z}{\text{médiane}} \; \bar{x}_{s,k,t}$$

- la moyenne par station évite qu'une station mesurée trois fois dans l'année pèse trois fois plus qu'une autre ;
- la médiane entre stations est moins sensible que la moyenne aux valeurs extrêmes de l'IPR ;
- `nb_stations` compte les stations de la zone qui ont au moins un indice cette année-là.

On obtient la base eau, `eau_zones_hydro.parquet` : une ligne par zone suivie et par année.

In [ ]:
eau_zones_hydro = agreger_eau_zones_hydro(indices, stations_zones_hydro)
eau_zones_hydro.to_parquet("data/processed/eau_zones_hydro.parquet", index=False)

print("doublons sur la clé :", eau_zones_hydro.duplicated(subset=["code_zone", "annee"]).sum())
eau_zones_hydro.shape

### 1.3 Pesticides : `pesticides.parquet`

La Banque nationale des ventes de produits phytopharmaceutiques (BNV-D, publiée sur data.gouv.fr) donne les quantités de substance active achetées chaque année, en kg, par département de l'acheteur, sur 2013-2023.

La base compte 511 substances. On en suit 10, choisies dans la littérature pour leur effet sur les cours d'eau et leur statut réglementaire (notebook [`tf_selection_molecules_eau`](notebooks/exploration/tf_selection_molecules_eau.ipynb)) :

| Famille | Molécules |
|---|---|
| Herbicides | glyphosate, S-métolachlore, isoproturon, chlorotoluron |
| Fongicides | cuivre, boscalid, tébuconazole |
| Insecticides | imidaclopride, cyperméthrine, chlorpyriphos-éthyl |

`preparer_pesticides` regroupe les 8 sels de cuivre sous « cuivre », retire la Corse et les DROM, et additionne les produits de biocontrôle et les autres : une ligne par département, année et molécule.

In [ ]:
pesticides = preparer_pesticides()

print("doublons sur la clé :", pesticides.duplicated(subset=["code_dep", "annee", "substance"]).sum())
pesticides.shape

### 1.4 Agriculture : `parcelles_rpg_2023.parquet`

Le Registre parcellaire graphique (RPG) recense les parcelles déclarées à la PAC, avec leur culture. On utilise l'archive nationale du RPG 2023 publiée par l'IGN, réduite à une ligne par parcelle avec son département, sa surface et son groupe de cultures (`construire_parcelles_rpg`). Les parcelles de surface nulle sont retirées.

L'assolement change peu d'une année à l'autre : le RPG 2023 sert pour toutes les années. `construire_parcelles_rpg` enchaîne quatre étapes :

1. **Lecture** de l'archive par paquets de 500 000 parcelles, pour tenir en mémoire (`lire_parcelles_rpg`).
2. **Département** (`rattacher_departements`) : l'archive ne le donne pas, `ID_PARCEL` est un simple numéro. Chaque parcelle va au département qui contient un point de sa surface (`representative_point`, toujours à l'intérieur du polygone, contrairement au centroïde d'une parcelle en forme de L), ou au plus proche à moins de 1 km, car les contours des départements sont simplifiés. Les 45 202 parcelles de Corse sont ainsi écartées.
3. **Nettoyage** (`nettoyer_parcelles_rpg`) : aucun identifiant n'est en double ; environ 8 500 parcelles de surface nulle (des miettes de moins de 0,005 ha) sont retirées.
4. **Libellés** des groupes et catégories de cultures (`ajouter_libelles_rpg`) : l'archive ne donne que les codes.

Ces étapes demandent l'archive (2,75 Go) et plus de 5 minutes : la table qui en sort est lue dans la release (104 Mo). La partie 3.1 rejoue le calcul géométrique sur un département pour le montrer.

In [ ]:
parcelles = pd.read_parquet(
    recuperer_release("parcelles_rpg_2023.parquet"),
    columns=["id_parcelle", "code_dep", "surface_ha", "code_groupe", "libelle_groupe", "categorie_culture"],
)

print("parcelles :", len(parcelles))
print("surface agricole (millions d'ha) :", round(parcelles["surface_ha"].sum() / 1e6, 2))

## 2. Statistiques descriptives de chaque base

### 2.1 Qualité de l'eau (`eau_zones_hydro`)

#### 2.1.1 Distribution des indices (2013-2023)

Valeurs de chaque indice par zone et par année (`eau_zones_hydro`, partie 1.2).

In [ ]:
eau_zones_hydro[["I2M2", "IBD", "IPR"]].describe().round(2)

Les trois indices sont très dispersés d'une zone à l'autre : l'I2M2 couvre toute l'échelle de 0 à 1. L'IPR a quelques valeurs extrêmes (jusqu'à 165) : on s'appuie donc sur la **médiane** plutôt que sur la moyenne.

#### 2.1.2 Évolution nationale 2013-2023

Pour chaque année, médiane des zones suivies cette année-là (`eau_zones_hydro`). L'axe de l'IPR est inversé : sur les trois panneaux, **le haut signifie une meilleure qualité**.

In [ ]:
tracer_evolution_indices(eau_zones_hydro);

Les variations sont faibles (attention, les axes ne partent pas de 0) : la médiane de l'I2M2 reste entre 0,53 et 0,61, celle de l'IBD entre 15,1 et 15,5. L'I2M2 baisse depuis 2017 ; l'IPR se dégrade, avec un saut en 2017 (de 15,4 en 2013 à 16,4 en 2023) ; l'IBD est stable. Les zones suivies ne sont pas toutes les mêmes d'une année à l'autre, ce qui peut suffire à créer des écarts de cette taille, comme le saut de 2017.

#### 2.1.3 Qualité biologique par zone hydrographique

Valeur de chaque indice par zone (`eau_zones_hydro`, partie 1.2), pour une année : 2023 par défaut, l'année du RPG utilisé en partie 3. Pour voir une autre année, il suffit de changer `ANNEE_CARTE`. La palette de l'IPR est inversée : sur les trois cartes, **foncé = meilleure qualité**. Chaque carte a sa propre échelle, du 2e au 98e centile. Les zones sans station suivie cette année-là sont en gris.

In [ ]:
ANNEE_CARTE = 2023
eau_annee = eau_zones_hydro[eau_zones_hydro["annee"] == ANNEE_CARTE]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
tracer_carte(zones_hydro, eau_annee, "I2M2", f"I2M2 (invertébrés), {ANNEE_CARTE}", axes[0])
tracer_carte(zones_hydro, eau_annee, "IBD", f"IBD (diatomées), {ANNEE_CARTE}", axes[1])
tracer_carte(zones_hydro, eau_annee, "IPR", f"IPR (poissons), {ANNEE_CARTE}", axes[2], cmap="Blues_r")

Pour l'I2M2 et l'IBD, la qualité est meilleure dans les massifs (Massif central, Alpes, Pyrénées) et plus faible dans le Bassin parisien et le Centre-Ouest. Mais la qualité varie fortement **à l'intérieur** d'un même département, d'une zone à sa voisine : c'est ce qui justifie le croisement avec les cultures à cette maille (partie 3).

Limites, pour 2023 :
- environ 3 400 des 6 110 zones ont au moins une station suivie cette année-là, et l'IPR, moins suivi, n'est disponible que pour environ 1 400 zones : sa carte est difficile à lire ;
- dans deux zones suivies sur trois, la valeur de l'année repose sur une seule station.

#### 2.1.4 Évolution 2013-2023 par zone hydrographique

Taux d'évolution de chaque indice entre la première et la dernière année, en %, pour les zones suivies ces deux années-là. Les palettes sont orientées pour que, sur les trois cartes, **bleu = amélioration** et **rouge = dégradation** : pour l'IPR, une hausse est une dégradation.

In [ ]:
ANNEE_DEBUT, ANNEE_FIN = 2013, 2023
debut = eau_zones_hydro[eau_zones_hydro["annee"] == ANNEE_DEBUT].set_index("code_zone")[["I2M2", "IBD", "IPR"]]
fin = eau_zones_hydro[eau_zones_hydro["annee"] == ANNEE_FIN].set_index("code_zone")[["I2M2", "IBD", "IPR"]]
evolution = ((fin - debut) / debut.where(debut > 0) * 100).reset_index()  # pas de taux si la valeur de départ est nulle
print("zones avec un taux :", evolution[["I2M2", "IBD", "IPR"]].count().to_dict())

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
tracer_carte(zones_hydro, evolution, "I2M2", f"I2M2, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[0], cmap="RdBu", centrer=True)
tracer_carte(zones_hydro, evolution, "IBD", f"IBD, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[1], cmap="RdBu", centrer=True)
tracer_carte(zones_hydro, evolution, "IPR", f"IPR, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[2], cmap="RdBu_r", centrer=True)

Il n'y a pas de tendance d'ensemble : entre 2013 et 2023, l'I2M2 progresse dans environ la moitié des zones, l'IBD dans 46 % d'entre elles, et l'IPR s'améliore dans 40 % seulement. Aucune région ne se détache nettement, les zones qui s'améliorent et celles qui se dégradent sont souvent voisines.

Ces cartes sont à lire avec prudence :
- seules les zones suivies en 2013 **et** en 2023 ont un taux (environ 1 400 pour l'I2M2, 2 200 pour l'IBD, 1 000 pour l'IPR). Pour l'I2M2, l'ouest de la France manque presque entièrement : le réseau n'a été étendu qu'en 2015 ;
- chaque taux compare deux années isolées, souvent une seule mesure par zone : une partie des écarts est du bruit d'une année sur l'autre.

### 2.2 Achats de pesticides (`pesticides`)

#### 2.2.1 Achats des 10 molécules (2013-2023)

Quantités achetées chaque année en métropole continentale, en tonnes de substance active. Chaque panneau a sa propre échelle.

In [ ]:
achats = pesticides.pivot_table(index="annee", columns="substance", values="quantite_kg", aggfunc="sum", fill_value=0) / 1000

axes = achats.plot(subplots=True, layout=(2, 5), figsize=(15, 5.5), color="#2a78d6", legend=False, title=list(achats.columns), xticks=range(2013, 2024, 2))
axes[0, 0].figure.tight_layout()

- Le glyphosate pèse à lui seul la moitié des achats des 10 molécules (6 700 t sur 13 200 t en 2023).
- 2013 est anormalement bas pour toutes les molécules (6 100 t en tout, contre 10 000 à 17 000 t les années suivantes). Aucun retrait ne l'explique : c'est sans doute une première année de collecte incomplète. Les évolutions sont donc à lire à partir de 2014.
- Le pic de 2018, suivi d'un creux en 2019, correspond probablement à des achats anticipés avant la hausse de la redevance pour pollutions diffuses au 1er janvier 2019.
- Trois molécules retirées du marché disparaissent presque des achats : l'isoproturon et le chlorpyriphos-éthyl dès 2017, l'imidaclopride dès 2018.
- Le S-métolachlore atteint son maximum en 2023 (2 800 t).

#### 2.2.2 Répartition des achats entre départements

Achats moyens par an de chaque molécule et par département, sur 2014-2023 (2013 est incomplète). Pour chaque molécule : la part des 10 premiers départements, le premier d'entre eux, et l'indice de Gini (`calculer_gini`), qui résume la concentration en un seul nombre : 0 si les achats sont répartis également entre les départements, proche de 1 s'ils sont concentrés dans quelques-uns.

In [ ]:
DEP_SANS_SURFACE = ["75", "92", "93", "94"]  # Paris et petite couronne : presque aucune surface agricole
moyenne_dep = pesticides[pesticides["annee"] >= 2014].pivot_table(
    index="code_dep", columns="substance", values="quantite_kg", aggfunc="sum", fill_value=0,
) / 10  # kg par an, en moyenne sur les 10 années 2014-2023
noms_dep = pesticides.drop_duplicates("code_dep").set_index("code_dep")["nom_dep"]

pd.DataFrame({
    "tonnes_par_an": moyenne_dep.sum() / 1000,
    "gini": moyenne_dep.apply(calculer_gini),
    "part_10_premiers_dep_pct": moyenne_dep.apply(lambda s: s.nlargest(10).sum() / s.sum() * 100),
    "premier_dep": moyenne_dep.idxmax().map(noms_dep),
}).sort_values("gini").round(2)

In [ ]:
ordre_mediane = moyenne_dep.median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=np.log10(moyenne_dep[ordre_mediane] + 1), color="#2a78d6", ax=ax)
ax.set(xlabel="", ylabel="log10(kg par an + 1)")
ax.set_title("Achats moyens par an et par département, 2014-2023")
ax.tick_params(axis="x", rotation=30);

Le boxplot montre l'écart entre départements, sur une échelle logarithmique : le département médian achète 76 t de glyphosate par an et 15 t de S-métolachlore, mais moins d'une tonne d'imidaclopride ou de chlorpyriphos-éthyl. Le quart des départements qui achètent le plus achète 4 fois plus de glyphosate que le quart qui achète le moins ; ce rapport monte à 15 pour le cuivre et 35 pour l'isoproturon, des molécules achetées dans quelques régions seulement.

Le glyphosate, utilisé sur presque toutes les cultures, est le plus également réparti (Gini de 0,40 ; les 10 premiers départements font 24 % des achats). À l'opposé, le cuivre est très concentré (Gini de 0,73 ; 59 % des achats dans 10 départements) : c'est le fongicide de la vigne, acheté surtout en Gironde, dans le Gard et l'Hérault. Les molécules des céréales (chlorotoluron, isoproturon, tébuconazole, imidaclopride) sont achetées d'abord dans les plaines céréalières du Bassin parisien et du Nord (Eure-et-Loir, Eure, Aisne, Somme, Marne), et le S-métolachlore, herbicide du maïs, dans le Sud-Ouest (Landes, Gers, Lot-et-Garonne).

#### 2.2.3 Où chaque molécule est achetée

Achats moyens par an sur 2014-2023, par département (`moyenne_dep`, partie 2.2.2), une carte par molécule. Les quantités varient de quelques kilos à plusieurs centaines de tonnes selon les départements : on représente $\log_{10}(\text{kg} + 1)$, et chaque carte a sa propre échelle. Le détail de l'exploration est dans le notebook [`notebook_pesticides_final`](notebooks/exploration/notebook_pesticides_final.ipynb).

In [ ]:
ordre_molecules = moyenne_dep.sum().sort_values(ascending=False).index  # de la plus achetée à la moins achetée
log_achats = np.log10(moyenne_dep + 1).rename_axis("code_dep").reset_index()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, molecule in zip(axes.flat, ordre_molecules):
    tracer_carte(departements, log_achats, molecule, molecule, ax, cmap="YlOrRd", cle="code_dep")
fig.suptitle("Achats moyens par an, 2014-2023 : log10(kg + 1)")
fig.tight_layout()

Chaque molécule dessine la carte de ses cultures :
- le **glyphosate** est acheté partout, sauf dans les massifs d'élevage (Massif central, Alpes) ;
- les molécules des **céréales** (chlorotoluron, isoproturon, tébuconazole, boscalid, imidaclopride) se concentrent dans la moitié nord, du Bassin parisien à la Lorraine ;
- le **S-métolachlore**, herbicide du maïs, ressort dans le Sud-Ouest, en Alsace et dans l'Ouest ;
- le **cuivre** suit la vigne : Bordelais, Languedoc, vallée du Rhône, Provence.

#### 2.2.4 Molécule la plus achetée par département

Pour chaque département, la molécule dont les achats moyens sur 2014-2023 sont les plus élevés. Le glyphosate domine presque partout : la carte de droite l'écarte pour faire apparaître la deuxième molécule, plus révélatrice des cultures du département.

In [ ]:
molecule_dominante = pd.DataFrame({
    "toutes": moyenne_dep.idxmax(axis=1),
    "hors_glyphosate": moyenne_dep.drop(columns="glyphosate").idxmax(axis=1),
}).rename_axis("code_dep").reset_index()
carte_dominante = departements.merge(molecule_dominante, on="code_dep", how="left")
categories = sorted(set(molecule_dominante["toutes"]) | set(molecule_dominante["hors_glyphosate"]))  # mêmes couleurs sur les deux cartes

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, colonne, titre in zip(axes, ["toutes", "hors_glyphosate"], ["Molécule la plus achetée", "Molécule la plus achetée hors glyphosate"]):
    carte_dominante.plot(
        column=colonne, categorical=True, categories=categories, cmap="tab10", ax=ax,
        edgecolor="white", linewidth=0.3, legend=True, legend_kwds={"loc": "lower left", "fontsize": 8},
    )
    ax.set_title(f"{titre}, 2014-2023")
    ax.set_axis_off()
print(molecule_dominante["hors_glyphosate"].value_counts().to_dict())

Le glyphosate est la molécule la plus achetée dans 84 des 94 départements. Les exceptions sont des départements de vigne, où c'est le cuivre (Gironde, Gard, Vaucluse, Var, Hautes-Alpes), et de maïs, où c'est le S-métolachlore (Landes, Pyrénées-Atlantiques, Hautes-Pyrénées, Bas-Rhin, Haut-Rhin).

Sans le glyphosate, la France se partage en trois grands profils de cultures :
- **maïs et polyculture-élevage** (S-métolachlore, 42 départements) : Ouest, Sud-Ouest, Centre et Alsace ;
- **grandes cultures** (chlorotoluron, 24 départements) : Bassin parisien, Centre, Poitou et Bourgogne-Franche-Comté, avec le tébuconazole (Aisne, Ardennes) et l'isoproturon (Moselle, Meurthe-et-Moselle) dans le Nord-Est ;
- **vigne** (cuivre, 24 départements) : façade méditerranéenne, vallée du Rhône, Bordelais, Charentes et Champagne.

Les molécules d'un même profil sont achetées aux mêmes endroits. Pour le croisement avec l'eau (parties 4 et 5), elles seront donc difficiles à séparer : un effet attribué à l'une peut venir d'une autre.

#### 2.2.5 Évolution des achats par département

Taux d'évolution des achats de chaque molécule entre la moyenne annuelle de **2014-2016** et celle de **2021-2023**. Comparer des moyennes sur trois ans plutôt que deux années isolées lisse le pic de 2018 et le creux de 2019 (partie 2.2.1). Les molécules quasi disparues du marché (baisse nationale de plus de 95 %) sont retirées : leur carte serait uniformément à -100 %. Les taux sont plafonnés à ±100 %, sinon quelques départements partis de presque rien écrasent l'échelle. **Rouge = hausse, bleu = baisse.**

In [ ]:
def achats_moyens(annees):
    """Achats moyens par an de chaque molécule et par département, en kg, sur les années données."""
    selection = pesticides[pesticides["annee"].isin(annees)]
    return selection.pivot_table(index="code_dep", columns="substance", values="quantite_kg", aggfunc="sum", fill_value=0) / len(annees)


achats_debut, achats_fin = achats_moyens([2014, 2015, 2016]), achats_moyens([2021, 2022, 2023])
evolution_nationale = (achats_fin.sum() / achats_debut.sum() - 1) * 100
print("évolution nationale (%) :", evolution_nationale.sort_values().round(0).to_dict())

molecules_vendues = evolution_nationale[evolution_nationale > -95].sort_values(ascending=False).index
evolution_achats = ((achats_fin / achats_debut.where(achats_debut > 0) - 1) * 100).clip(-100, 100).rename_axis("code_dep").reset_index()

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, molecule in zip(axes.flat, molecules_vendues):
    tracer_carte(departements, evolution_achats, molecule, f"{molecule} ({evolution_nationale[molecule]:+.0f} % en France)", ax, cmap="RdBu_r", centrer=True, cle="code_dep")
for ax in axes.flat[len(molecules_vendues):]:
    ax.set_axis_off()
fig.suptitle("Évolution des achats (%), moyenne 2021-2023 vs moyenne 2014-2016")
fig.tight_layout()

- Le **glyphosate** baisse de 14 % en France et recule dans 71 départements sur 90, surtout dans le Sud-Ouest et le Sud-Est (Gironde -61 %, Gard -56 %). Il progresse en revanche dans le nord-est du Bassin parisien (Ardennes +53 %, Meuse +37 %, Seine-et-Marne +31 %).
- Le **boscalid** recule partout (-75 % en France).
- Le **S-métolachlore** (+12 %) et la **cyperméthrine** (+5 %) progressent dans environ deux départements sur trois, surtout dans la moitié nord et l'Ouest.
- Le **chlorotoluron** est stable en France (+2 %), mais avec des évolutions opposées : hausse en Bretagne et en Champagne, baisse dans le Sud.

Les plus fortes hausses touchent souvent des départements qui achetaient peu au départ (S-métolachlore dans les Alpes-Maritimes ou en Lozère) : elles comptent peu dans le total national. Ces évolutions sont celles des achats, enregistrés à l'adresse de l'acheteur : un distributeur qui change de département peut déplacer des achats d'une carte à l'autre.

#### 2.2.6 Tendance des achats

Total des achats des 10 molécules chaque année, avec une droite de régression ajustée par moindres carrés sur 2014-2023 (2013 est incomplète) :

$$Q_t = a + b\,t + \varepsilon_t$$

La pente $b$ donne la variation moyenne des achats, en tonnes par an. Le $R^2$ mesure la part des variations d'une année à l'autre que la droite explique.

In [ ]:
total_annuel = pesticides.groupby("annee")["quantite_kg"].sum() / 1000  # tonnes
periode = total_annuel.loc[2014:]
pente, constante = np.polyfit(periode.index, periode.values, 1)
ajuste = constante + pente * periode.index
r2 = 1 - ((periode - ajuste) ** 2).sum() / ((periode - periode.mean()) ** 2).sum()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(total_annuel.index, total_annuel.values, marker="o", color="#2a78d6", label="achats")
ax.plot(periode.index, ajuste, linestyle="--", color="black", label=f"tendance 2014-2023 : {pente:+.0f} t/an (R² = {r2:.2f})")
ax.set_xticks(range(2013, 2024))
ax.set_ylabel("tonnes de substance active")
ax.set_title("Achats des 10 molécules")
ax.legend();

Les achats des 10 molécules baissent en moyenne de 430 t par an sur 2014-2023, soit environ 3 % de leur niveau moyen (13 900 t par an). Cette tendance est fragile : elle repose sur 10 années, et la droite n'explique que 39 % des variations ($R^2 = 0{,}39$), le reste venant surtout du pic de 2018 et du creux de 2019. La baisse vient du retrait de l'isoproturon, de l'imidaclopride et du chlorpyriphos-éthyl et du recul du glyphosate, en partie compensés par la hausse du S-métolachlore et du cuivre (partie 2.2.5).

### 2.3 Cultures (`parcelles`)

#### 2.3.1 National : grandes catégories, taille des parcelles et 23 groupes de cultures

Répartition nationale des 9,74 millions de parcelles et des 27,46 millions d'hectares du RPG 2023 entre les grandes catégories d'occupation du sol et les 23 groupes de cultures, avec la distribution des surfaces de parcelles (`surface_ha`).

In [ ]:
print("Distribution nationale de surface_ha :", parcelles["surface_ha"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(2).to_dict())

cat_propre = parcelles["categorie_culture"].map(CATEGORIES_CULTURE).fillna("Hors production (bordures, bandes tampons)")
grp_cat = parcelles.groupby(cat_propre)["surface_ha"]
quantiles_cat = grp_cat.quantile([0.25, 0.50, 0.75, 0.95]).unstack()

stats_categories = pd.DataFrame({
    "nb_parcelles": grp_cat.count(),
    "part_parcelles_pct": grp_cat.count() / len(parcelles) * 100,
    "surface_Mha": grp_cat.sum() / 1e6,
    "part_surface_pct": grp_cat.sum() / parcelles["surface_ha"].sum() * 100,
    "moyenne_ha": grp_cat.mean(),
    "mediane_ha": quantiles_cat[0.50],
    "q25_ha": quantiles_cat[0.25],
    "q75_ha": quantiles_cat[0.75],
    "q95_ha": quantiles_cat[0.95],
    "max_ha": grp_cat.max(),
}).sort_values("surface_Mha", ascending=False)

stats_categories.round(2)

In [ ]:
couleurs_cat = {
    "Terres arables": "#2a78d6",
    "Prairies permanentes": "#2ca25f",
    "Cultures permanentes": "#7b3294",
    "Hors production (bordures, bandes tampons)": "#969696",
}
part_groupes = parcelles.groupby("libelle_groupe")["surface_ha"].sum().sort_values() / parcelles["surface_ha"].sum() * 100
cat_par_groupe = cat_propre.groupby(parcelles["libelle_groupe"]).agg(lambda s: s.value_counts().index[0])

fig, ax = plt.subplots(figsize=(8, 6.2))
part_groupes.plot.barh(ax=ax, color=cat_par_groupe.reindex(part_groupes.index).map(couleurs_cat))
ax.set_xlabel("part de la surface agricole (%)")
ax.set_ylabel("")
ax.set_title("Part de la surface agricole par groupe de cultures (RPG 2023)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in couleurs_cat.values()]
ax.legend(handles, couleurs_cat.keys(), title="Catégorie dominante", loc="lower right", fontsize=8.5);

- **Taille nationale très asymétrique** : la parcelle médiane mesure **1,24 ha** (Q25 = 0,40 ha, Q75 = 3,21 ha, Q95 = 10,34 ha) pour une moyenne de **2,82 ha**, plus du double de la médiane.
- **Terres arables** : **60,8 %** de la surface (16,69 M ha) pour 48,2 % des parcelles, avec les plus grandes surfaces unitaires (médiane 2,00 ha, moyenne 3,55 ha).
- **Prairies permanentes** : **35,3 %** de la surface (9,68 M ha, médiane 1,20 ha, max 6 545 ha sur les grandes estives de montagne) ; avec les prairies temporaires et le fourrage, l'herbe couvre **44 %** de la SAU.
- **Cultures permanentes** (vignes, vergers, oliviers, fruits à coque) : seulement **3,1 %** de la surface (0,86 M ha) mais **7,6 %** des parcelles (médiane 0,63 ha), reflet d'un parcellaire morcelé et fortement traité.
- **Hors production** : près de 955 000 parcelles (**9,8 %** du total) correspondent à des bordures, bandes tampons ou surfaces non agricoles, pour **0,8 %** de la surface (médiane 0,10 ha).
- **Groupes de cultures** : l'herbe domine, suivie par les grandes cultures — blé tendre (17,2 %), maïs (9,4 %), orge (6,6 %), colza (4,9 %) et tournesol (3,0 %).

#### 2.3.2 Filières : morphologie parcellaire et concentration géographique

Les 23 groupes sont réunis en **8 grandes filières agronomiques** (`FILIERES_RPG`). Pour chacune, on compare la taille des parcelles (`surface_ha`) et la concentration entre départements avec l'indice de Gini de la partie 2.2.2 (0 = répartie également, proche de 1 = concentrée dans quelques départements).

In [ ]:
parcelles["filiere"] = parcelles["libelle_groupe"].map(FILIERES_RPG)
surface_dep_filiere = parcelles.pivot_table(index="code_dep", columns="filiere", values="surface_ha", aggfunc="sum", fill_value=0)
grp_fil = parcelles.groupby("filiere")["surface_ha"]

stats_filieres = pd.DataFrame({
    "surface_Mha": grp_fil.sum() / 1e6,
    "part_sau_pct": grp_fil.sum() / parcelles["surface_ha"].sum() * 100,
    "nb_parcelles": grp_fil.count(),
    "mediane_ha": grp_fil.median(),
    "moyenne_ha": grp_fil.mean(),
    "gini_dep": surface_dep_filiere.apply(calculer_gini),
    "part_10_premiers_dep_pct": surface_dep_filiere.apply(lambda s: s.nlargest(10).sum() / s.sum() * 100),
    "premier_dep": surface_dep_filiere.idxmax().map(noms_dep),
}).sort_values("surface_Mha", ascending=False)

stats_filieres.round(2)

In [ ]:
tracer_filieres_rpg(parcelles, stats_filieres);

Deux contrastes majeurs se dégagent :

- **Taille des parcelles** : les céréales à paille (médiane 3,0 ha, moyenne 4,8 ha) et les oléagineux et cultures industrielles (médiane 3,5 ha, moyenne 5,3 ha) s'organisent en grandes parcelles d'openfield, cinq fois plus grandes en médiane que les vignes et vergers (0,62 ha) ou les légumes et fleurs (0,71 ha).
- **Concentration territoriale** : les prairies permanentes (Gini 0,41) et les céréales à paille (0,48) couvrent une grande partie du territoire, tandis que les **vignes et vergers** (Gini **0,76** ; 62 % de la surface dans 10 départements, Gironde en tête) et les **légumes et fleurs** (0,73 ; Somme) sont très localisés. Ce Gini des vignes et vergers fait directement écho à celui du cuivre en partie 2.2.2 (0,73).

#### 2.3.3 Disparités régionales : surfaces, taille des parcelles et spécialisation

Agrégation aux **12 régions métropolitaines continentales** (`DEPARTEMENT_VERS_REGION`, hors Corse) : poids agricole de chaque région, morphologie de son parcellaire et répartition de sa SAU entre les 8 filières.

In [ ]:
region_parcelle = parcelles["code_dep"].map(DEPARTEMENT_VERS_REGION).rename("region")
grp_reg = parcelles.groupby(region_parcelle)["surface_ha"]
surface_reg_fil = parcelles.pivot_table(index=region_parcelle, columns="filiere", values="surface_ha", aggfunc="sum", fill_value=0)
parts_reg_fil = surface_reg_fil.div(surface_reg_fil.sum(axis=1), axis=0) * 100

stats_regions = pd.DataFrame({
    "surface_Mha": grp_reg.sum() / 1e6,
    "part_sau_pct": grp_reg.sum() / parcelles["surface_ha"].sum() * 100,
    "nb_parcelles": grp_reg.count(),
    "mediane_ha": grp_reg.median(),
    "moyenne_ha": grp_reg.mean(),
    "filiere_dominante": parts_reg_fil.idxmax(axis=1),
    "part_dominante_pct": parts_reg_fil.max(axis=1),
}).sort_values("surface_Mha", ascending=False)

stats_regions.round({"surface_Mha": 2, "part_sau_pct": 1, "mediane_ha": 2, "moyenne_ha": 2, "part_dominante_pct": 1})

In [ ]:
tracer_regions_rpg(stats_regions, parts_reg_fil);

L'échelle régionale fait ressortir trois grands ensembles :

- **Grandes plaines de cultures annuelles** : **Île-de-France** (79 % de céréales à paille et oléagineux/indus., parcelle moyenne de 4,5 ha — la plus élevée de France), **Hauts-de-France** (68 %, plus 7 % de légumes et fleurs), **Centre-Val de Loire** (68 %, moy. 4,0 ha) et **Grand Est** (55 %, 2,98 M ha).
- **Polyculture-élevage de l'Ouest** : **Bretagne** (27 % de maïs, 27 % de céréales, 35 % de prairies et fourrages ; plus petite moyenne régionale à 1,9 ha), **Pays de la Loire** et **Normandie** (plus haute médiane régionale à 2,1 ha).
- **Massifs, élevage herbager et cultures permanentes** : **Nouvelle-Aquitaine** (1re SAU avec 3,87 M ha) et **Occitanie** (3,65 M ha) mêlent élevage, grandes cultures et vignes/vergers (6 à 7 % de leur SAU) ; **Auvergne-Rhône-Alpes** (64 % de prairies permanentes et estives) et **Provence-Alpes-Côte d'Azur** (71 % d'herbe et estives, 9 % de vignes/vergers) présentent les parcelles médianes les plus petites (1,0 ha et 0,7 ha), mais en PACA de vastes estives alpines tirent la moyenne à 3,2 ha.

#### 2.3.4 Géographie départementale des filières et du parcellaire

Part de la surface agricole départementale consacrée aux trois principales filières exposées aux traitements (grandes cultures hors maïs, maïs, vignes et vergers) et taille médiane des parcelles par département. Paris et la petite couronne sont laissés en gris.

In [ ]:
parts_dep_fil = surface_dep_filiere.div(surface_dep_filiere.sum(axis=1), axis=0) * 100
geo_rpg = pd.DataFrame({
    "grandes_cultures": parts_dep_fil["Céréales à paille"] + parts_dep_fil["Oléagineux & indus."],
    "mais": parts_dep_fil["Maïs"],
    "vignes_vergers": parts_dep_fil["Vignes & vergers"],
    "mediane_ha": parcelles.groupby("code_dep")["surface_ha"].median(),
}).drop(DEP_SANS_SURFACE).rename_axis("code_dep").reset_index()

fig, axes = plt.subplots(2, 2, figsize=(12, 10.5))
tracer_carte(departements, geo_rpg, "grandes_cultures", "Céréales à paille, oléagineux & indus. (% SAU)", axes[0, 0], cmap="YlOrBr", cle="code_dep")
tracer_carte(departements, geo_rpg, "mais", "Maïs grain et ensilage (% SAU)", axes[0, 1], cmap="Oranges", cle="code_dep")
tracer_carte(departements, geo_rpg, "vignes_vergers", "Vignes, vergers & cultures perm. (% SAU)", axes[1, 0], cmap="Purples", cle="code_dep")
tracer_carte(departements, geo_rpg, "mediane_ha", "Taille médiane des parcelles (ha)", axes[1, 1], cmap="Blues", cle="code_dep")
fig.tight_layout()

Les spécialisations départementales affinent le découpage régional :

- **Grandes cultures hors maïs** : l'arc du Bassin parisien, des Hauts-de-France, de Champagne et du Centre dépasse 78 à 83 % de la surface (Eure-et-Loir 83 %, Essonne 82 %, Seine-et-Marne 80 %, Marne 80 %).
- **Maïs** : deux bassins très spécialisés — le Sud-Ouest (Landes 58 %) et l'Alsace (Haut-Rhin 41 %, Bas-Rhin 32 %) — ainsi que l'Ouest laitier et porcin.
- **Vignes et vergers** : concentrés dans le Bordelais et les Charentes (Gironde 42 %) et sur l'arc méditerranéen (Vaucluse 39 %, Hérault 34 %, Gard, Aude).
- **Taille médiane des parcelles** : l'openfield du Bassin parisien et de Lorraine (médiane de 2,5 à 3,0 ha en Haute-Marne, Ardennes, Marne, Seine-Maritime et Eure) s'oppose au parcellaire très morcelé (0,5 à 0,6 ha de médiane) en Alsace, dans l'Hérault, la Drôme et les Hautes-Alpes.

#### 2.3.5 Zoom sur 8 départements contrastés

Comparaison de 8 départements représentatifs des grands systèmes agricoles : grandes cultures (Eure-et-Loir, Marne), maïsiculture (Landes), polyculture-élevage (Finistère), viticulture et arboriculture (Gironde, Vaucluse), élevage herbager et de montagne (Aveyron, Cantal).

In [ ]:
ZOOM_DEP = ["28", "51", "40", "29", "33", "84", "12", "15"]
achats_dep = pesticides[pesticides["annee"] == ANNEE_CARTE].groupby("code_dep")["quantite_kg"].sum()
surface_dep = parcelles.groupby("code_dep")["surface_ha"].sum()

zoom = pd.DataFrame({
    "departement": [noms_dep[c] for c in ZOOM_DEP],
    "surface_kha": surface_dep.reindex(ZOOM_DEP) / 1000,
    "nb_parcelles": parcelles.groupby("code_dep")["surface_ha"].count().reindex(ZOOM_DEP),
    "mediane_ha": parcelles.groupby("code_dep")["surface_ha"].median().reindex(ZOOM_DEP),
    "filiere_dominante": parts_dep_fil.loc[ZOOM_DEP].idxmax(axis=1),
    "part_dominante_pct": parts_dep_fil.loc[ZOOM_DEP].max(axis=1),
    "kg_par_ha_2023": (achats_dep / surface_dep).reindex(ZOOM_DEP),
}, index=ZOOM_DEP)

zoom.round({"surface_kha": 1, "mediane_ha": 2, "part_dominante_pct": 1, "kg_par_ha_2023": 2})

In [ ]:
tracer_zoom_departements(parcelles, pesticides, noms_dep, ZOOM_DEP, annee=ANNEE_CARTE);

Le lien entre filière dominante, taille des parcelles et pression phytosanitaire ressort nettement :

- **Élevage herbager** : dans le Cantal (87 % de prairies permanentes et estives) et l'Aveyron (2e SAU départementale avec 513 000 ha, dont 83 % d'herbe et fourrages), les achats des 10 molécules sont quasi nuls (0,02 et 0,05 kg/ha).
- **Polyculture-élevage** : le Finistère équilibre maïs (27 %), céréales (23 %), prairies permanentes (23 %) et prairies temporaires/fourrages (16 %), pour une intensité intermédiaire (0,39 kg/ha) et un parcellaire fin (médiane 1,26 ha).
- **Grandes cultures et maïs** : l'Eure-et-Loir et la Marne (1re SAU départementale, 530 000 ha) consacrent 80 à 83 % de leur surface aux céréales, oléagineux et betteraves (0,71 à 0,84 kg/ha, glyphosate en tête). Les Landes (58 % de maïs) atteignent 1,16 kg/ha, avec le S-métolachlore en première molécule.
- **Vignes et vergers** : en Gironde (42 % de vignes et vergers) et dans le Vaucluse (39 %, parcelles médianes de 0,61 ha), les achats atteignent 2,08 et 1,24 kg/ha, tirés par le cuivre.

**Limite de couverture du RPG (exemple de la Marne).** La Marne ne compte que **2 566 ha de vignes** (0,5 % de sa SAU) dans le RPG, très en deçà du vignoble champenois réel. Le RPG ne recense que les parcelles déclarées à la PAC : les exploitations viticoles spécialisées qui ne demandent pas d'aides PAC (fréquentes en Champagne, en Alsace ou en Bourgogne) y sont sous-représentées. Lors de la répartition par culture (partie 3.2), les achats de cuivre d'un tel département se concentrent donc sur les seules vignes déclarées.

#### 2.3.6 Cultures et achats de pesticides par département

À gauche, la part de la surface agricole en prairies permanentes et estives. À droite, les achats des 10 molécules rapportés à la surface agricole du département, en 2023 (`ANNEE_CARTE`, partie 2.1.3).

In [ ]:
herbe = parcelles["libelle_groupe"].isin(["Prairies permanentes", "Estives et landes"])
achats_dep_carte = achats_dep.drop(DEP_SANS_SURFACE)  # achats sans surface agricole : en gris
par_dep = pd.DataFrame({
    "part_herbe": parcelles[herbe].groupby("code_dep")["surface_ha"].sum() / surface_dep * 100,
    "kg_par_ha": achats_dep_carte / surface_dep,
}).rename_axis("code_dep").reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
tracer_carte(departements, par_dep, "part_herbe", "Prairies permanentes et estives (% de la surface agricole)", axes[0], cmap="Greens", cle="code_dep")
tracer_carte(departements, par_dep, "kg_par_ha", f"Achats des 10 molécules, {ANNEE_CARTE} (kg par ha agricole)", axes[1], cmap="Oranges", cle="code_dep")

Les deux cartes se répondent. Dans les massifs (Massif central, Alpes, Pyrénées, Jura), l'herbe couvre plus des trois quarts de la surface agricole et les achats sont presque nuls : 0,01 kg/ha en Lozère, 0,02 dans le Cantal. Ils sont les plus élevés dans les départements de vigne et de vergers (Gironde, 2,1 kg/ha ; Vaucluse, Lot-et-Garonne, Charente-Maritime), de maïs (Landes) et de grandes cultures du Bassin parisien (Val-d'Oise, Eure, Seine-et-Marne, Oise : environ 0,9 kg/ha). En moyenne nationale, on achète 0,48 kg de ces 10 molécules par hectare agricole.

Les achats sont localisés à l'adresse de l'acheteur, pas de la parcelle. À Paris et dans les Hauts-de-Seine, quelques tonnes rapportées à presque aucune surface agricole donnent des valeurs absurdes (50 000 et 900 kg/ha) : Paris et la petite couronne sont laissés en gris.

## 3. Fusion et agrégation à la zone hydrographique

La base eau est déjà à la maille de la **zone hydrographique** (partie 1.2). On y ramène les deux autres : les parcelles sont rattachées aux zones, ce qui donne la surface de chaque groupe de cultures par zone (3.1) ; les achats de pesticides du département sont répartis entre les parcelles, puis sommés par zone (3.2). La fusion des trois donne la base finale (3.3).

Seul `parcelles_zones_hydro.parquet`, long à calculer, est lu dans la release ; son calcul est rejoué sur un département pour le montrer (3.1). Le reste se recalcule en quelques secondes.

Notations de toute la partie 3 :

| Symbole | Signification |
|---|---|
| $i$ | parcelle du RPG 2023, de surface déclarée $s_i$ (ha), de département $d_i$ et de groupe de cultures $g_i$ |
| $z$ | zone hydrographique, de surface $A_z$ (ha) |
| $t$, $m$ | année, molécule |
| $Q_{d,t,m}$ | achats de la molécule $m$ dans le département $d$ l'année $t$, en kg (`pesticides`) |

### 3.1 Parcelles et cultures par zone hydrographique

Les parcelles viennent de l'archive nationale du RPG 2023 publiée par l'IGN, qui donne le contour de chaque parcelle (`associer_parcelles_zones_hydro`). Pour chaque parcelle $P_i$ et chaque zone $Z_z$ qu'elle touche :

$$\text{part}_{i,z} = \frac{\text{aire}(P_i \cap Z_z)}{\sum_{z'} \text{aire}(P_i \cap Z_{z'})}$$

- Une parcelle entièrement dans une zone y va entière ($\text{part}_{i,z} = 1$). L'intersection n'est calculée que pour les parcelles qui touchent plusieurs zones, moins de 5 % d'entre elles.
- Le dénominateur est l'aire de la parcelle qui tombe dans des zones, pas son aire totale : sur le littoral, le morceau qui déborde de toutes les zones est réparti entre les zones touchées au lieu d'être perdu. Les parts d'une parcelle font donc toujours 1.
- Les contours invalides (4 dans l'archive) sont réparés avant l'intersection (`make_valid`), et une zone qui ne touche la parcelle que par un bord (aire nulle) est écartée.

Sur la France entière, le calcul prend environ 25 minutes en plus du téléchargement de l'archive : la table est lue dans la release.

In [ ]:
parcelles_zones_hydro = pd.read_parquet(recuperer_release("parcelles_zones_hydro.parquet"))

print("parcelles :", parcelles_zones_hydro["id_parcelle"].nunique())
print("parcelles partagées entre plusieurs zones :", parcelles_zones_hydro.loc[parcelles_zones_hydro["part"] < 1, "id_parcelle"].nunique())

**Démonstration sur un département : l'Essonne.** Pour voir ce calcul tourner sans l'archive, la release contient les contours des 19 088 parcelles de l'Essonne (`parcelles_rpg_2023_91.gpkg`, 10 Mo), extraits de l'archive par `extraire_contours_departement`. On leur applique la même fonction `associer_parcelles_zones_hydro` (environ une seconde), puis on compare le résultat aux lignes de l'Essonne dans la table nationale.

In [ ]:
chemin_91 = recuperer_release("parcelles_rpg_2023_91.gpkg")
contours_91 = gpd.read_file(chemin_91)
parcelles_zones_91 = associer_parcelles_zones_hydro(chemin_91, zones_hydro)

release_91 = parcelles_zones_hydro[parcelles_zones_hydro["id_parcelle"].isin(contours_91["ID_PARCEL"])]
comparaison = parcelles_zones_91.merge(release_91, on=["id_parcelle", "code_zone"], how="outer", suffixes=("_demo", "_release"))

print("parcelles :", parcelles_zones_91["id_parcelle"].nunique(), "; coupées par une limite :", parcelles_zones_91.loc[parcelles_zones_91["part"] < 1, "id_parcelle"].nunique())
print("lignes présentes d'un seul côté :", comparaison[["part_demo", "part_release"]].isna().any(axis=1).sum())
print("écart maximal entre les parts :", (comparaison["part_demo"] - comparaison["part_release"]).abs().max())

Les deux tables sont identiques : la table de la release est bien celle que produit le code.

La carte montre les parcelles coupées (en rouge) et, à droite, un zoom de 3 km autour d'une parcelle de blé tendre (contour noir).

In [ ]:
EXEMPLE = "5279955"  # parcelle de blé tendre de 50 ha, à cheval sur deux zones
centre = contours_91.loc[contours_91["ID_PARCEL"] == EXEMPLE].geometry.iloc[0].centroid
cadre = (centre.x - 1500, centre.y - 1500, centre.x + 1500, centre.y + 1500)

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
tracer_parcelles_coupees(contours_91, parcelles_zones_91, zones_hydro, "Essonne : parcelles coupées par une limite de zone", axes[0])
tracer_parcelles_coupees(contours_91, parcelles_zones_91, zones_hydro, "Zoom autour de la parcelle d'exemple", axes[1], cadre=cadre)
contours_91[contours_91["ID_PARCEL"] == EXEMPLE].boundary.plot(ax=axes[1], color="black", linewidth=1.5);

In [ ]:
parcelles_zones_91[parcelles_zones_91["id_parcelle"] == EXEMPLE].merge(zones_hydro[["code_zone", "nom_zone"]], on="code_zone")

Les limites des zones suivent les lignes de partage des eaux entre deux cours d'eau ; les parcelles coupées se concentrent le long de ces lignes. La parcelle d'exemple, 50,5 ha de blé tendre, est partagée entre la zone de la Juine (47 %) et celle de la Challouette (53 %) : sa surface et ses pesticides seront répartis entre les deux zones dans ces proportions.

La surface de chaque groupe de cultures $g$ dans chaque zone $z$ en découle (`agreger_champs_zones_hydro`, quelques secondes, sans fichier) : chaque parcelle compte dans chaque zone pour sa part.

$$S_{z,g} = \sum_{i \,:\, g_i = g} \text{part}_{i,z} \, s_i$$

On prend la surface déclarée $s_i$ (`SURF_PARC` dans le RPG) plutôt que l'aire du contour. Sur le littoral, une parcelle qui déborde du contour de la zone y est comptée entière, si bien que la part agricole d'une zone côtière peut dépasser 100 % de quelques dixièmes.

In [ ]:
champs_zones_hydro = agreger_champs_zones_hydro(parcelles, parcelles_zones_hydro)

print("doublons sur la clé :", champs_zones_hydro.duplicated(subset=["code_zone", "code_groupe_culture"]).sum())
print("surface rattachée (millions d'ha) :", round(champs_zones_hydro["surface_ha"].sum() / 1e6, 2))

### 3.2 Pesticides par zone hydrographique

Les achats sont connus par département et par année, pas par parcelle. Pour les ramener à la zone, Tomia les répartit entre les parcelles qui ont pu les recevoir, en trois étapes (méthode détaillée dans [`docs/repartition_pesticides.md`](docs/repartition_pesticides.md)).

**Étape 1 : quelles cultures utilisent quelle molécule.** $I_{g,m} = 1$ si le groupe de cultures $g$ utilise la molécule $m$, 0 sinon (`MOLECULES_PAR_GROUPE`), d'après les homologations (E-Phy de l'Anses) et les guides des instituts techniques (notebook [`tf_correspondance_groupes_molecules`](notebooks/exploration/tf_correspondance_groupes_molecules.ipynb)). Quatre groupes n'en utilisent aucune : prairies permanentes, estives et landes, gel et divers, soit 38 % de la surface agricole.

In [ ]:
utilisation = pd.DataFrame({groupe: {m: "x" for m in molecules} for groupe, molecules in MOLECULES_PAR_GROUPE.items()}).T
utilisation.reindex(columns=MOLECULES).fillna("")

**Étape 2 : une dose par hectare** (`calculer_doses_pesticides_groupes`). Dans le département $d$, la surface éligible à la molécule $m$ est celle des groupes qui l'utilisent :

$$S^{\text{élig}}_{d,m} = \sum_g I_{g,m} \, S_{d,g} \qquad \text{avec} \qquad S_{d,g} = \sum_{i \,:\, d_i = d,\ g_i = g} s_i$$

Faute de savoir quelle culture a reçu quoi, les achats $Q_{d,t,m}$ sont répartis entre les parcelles éligibles au prorata de leur surface : chaque hectare éligible du département reçoit la même dose.

$$D_{d,t,m,g} = I_{g,m} \, \frac{Q_{d,t,m}}{S^{\text{élig}}_{d,m}} \qquad\qquad q_{i,t,m} = D_{d_i,t,m,g_i} \, s_i$$

- **Conservation** : les quantités des parcelles d'un département redonnent ses achats, $\sum_{i \,:\, d_i = d} q_{i,t,m} = Q_{d,t,m}$.
- **Exception** : un département sans surface éligible (Paris pour les 10 molécules, les Hauts-de-Seine pour le S-métolachlore et le cuivre) a une dose nulle ; ces achats, environ 136 t sur 2013-2023, ne sont attribués à aucune parcelle.
- **Hypothèse** : un blé et une orge du même département reçoivent la même dose de boscalid, et l'agriculture biologique n'est pas distinguée. La méthode ne fait que localiser les achats, à l'intérieur du département, là où se trouvent les cultures concernées.

In [ ]:
doses = calculer_doses_pesticides_groupes(parcelles, pesticides)

doses.head()

**Exemple** : le glyphosate en Essonne en 2023, et la parcelle d'exemple de la partie 3.1.

In [ ]:
DEP, MOLECULE, ANNEE = "91", "glyphosate", 2023
q = pesticides.loc[(pesticides["code_dep"] == DEP) & (pesticides["substance"] == MOLECULE) & (pesticides["annee"] == ANNEE), "quantite_kg"].sum()
groupes = [groupe for groupe, molecules in MOLECULES_PAR_GROUPE.items() if MOLECULE in molecules]
s_elig = parcelles.loc[(parcelles["code_dep"] == DEP) & parcelles["libelle_groupe"].isin(groupes), "surface_ha"].sum()
s_exemple = parcelles.loc[parcelles["id_parcelle"] == EXEMPLE, "surface_ha"].iloc[0]

print(f"Q = {q:,.0f} kg achetés ; S_élig = {s_elig:,.0f} ha dans les {len(groupes)} groupes qui l'utilisent")
print(f"D = Q / S_élig = {q / s_elig:.3f} kg/ha")
print(f"parcelle d'exemple : q = D × {s_exemple:.1f} ha = {q / s_elig * s_exemple:.1f} kg")

Les 24 600 kg de glyphosate achetés en Essonne en 2023 sont répartis sur 76 900 ha, 93 % de la surface agricole du département, car presque tous les groupes de cultures en utilisent : 0,32 kg/ha. La parcelle d'exemple en reçoit 16,2 kg.

**Étape 3 : de la parcelle à la zone** (`agreger_pesticides_zones_hydro`, quelques secondes, sans fichier). La zone reçoit la quantité de chaque parcelle multipliée par sa part :

$$Q_{z,t,m} = \sum_i \text{part}_{i,z} \, q_{i,t,m} = \sum_{d,g} D_{d,t,m,g} \sum_{i \,:\, d_i = d,\ g_i = g} \text{part}_{i,z} \, s_i$$

La seconde écriture est celle du code : la dose ne dépendant que du département, du groupe et de l'année, on somme d'abord les surfaces de la zone par département et par groupe, puis on les multiplie par les doses. On évite ainsi de croiser 9,7 millions de parcelles avec 11 années et 10 molécules. La parcelle d'exemple donne ainsi 47 % de ses 16,2 kg de glyphosate (7,6 kg) à la zone de la Juine et 53 % (8,6 kg) à celle de la Challouette.

In [ ]:
pesticides_zones_hydro = agreger_pesticides_zones_hydro(doses, parcelles, parcelles_zones_hydro)

print("doublons sur la clé :", pesticides_zones_hydro.duplicated(subset=["code_zone", "annee"]).sum())
pesticides_zones_hydro.groupby("annee")[LISTE_COLONNES_MOLECULES].sum().round(0)

**Contrôle.** La somme sur toutes les zones doit redonner les achats, aux pertes près : achats de Paris et des Hauts-de-Seine sans surface éligible, et quelques parcelles du littoral hors de toute zone.

In [ ]:
attribue = pesticides_zones_hydro[LISTE_COLONNES_MOLECULES].sum().sum()
print(f"achats retrouvés dans les zones : {attribue / pesticides['quantite_kg'].sum():.3%}")

### 3.3 Base finale : `base_finale.parquet`

Une ligne par zone suivie et par année (`fusionner_eau_cultures_pesticides`). Avec $A_z$ la surface de la zone (`surface_zone_ha`) :

| Colonnes | Calcul |
|---|---|
| `I2M2`, `IBD`, `IPR`, `nb_stations` | $I_{z,k,t}$, partie 1.2 |
| `part_groupe_<code>` | part de la zone occupée par le groupe de cultures, en % : $100 \, S_{z,g} / A_z$ |
| `part_agricole` | la même chose pour l'ensemble des groupes : $100 \, \sum_g S_{z,g} / A_z$ |
| `<molécule>_kg_ha` | pesticides attribués à la zone l'année de l'indice, par hectare de zone : $Q_{z,t,m} / A_z$ |

- On divise par la surface de la zone, et non par sa surface agricole, pour mesurer la pression qui s'exerce sur la rivière : à dose égale, une zone à moitié boisée reçoit deux fois moins de pesticides par hectare.
- Les cultures sont celles du RPG 2023, reprises pour toutes les années.
- Une zone suivie sans aucune parcelle a des parts et des pesticides nuls.

In [ ]:
base_finale = fusionner_eau_cultures_pesticides(eau_zones_hydro, champs_zones_hydro, pesticides_zones_hydro, zones_hydro)
base_finale.to_parquet("data/processed/base_finale.parquet", index=False)

print("doublons sur la clé :", base_finale.duplicated(subset=["code_zone", "annee"]).sum())
base_finale.shape

## 4. Statistiques descriptives de la base finale

_À choisir._

## 5. Interprétation

_À rédiger._

## 6. Conclusion et limites